In [1]:
import librosa
import IPython.display as ipd
import numpy as np
import soundfile as sf

In [2]:
filename = '../../data/tomsdiner_mono.wav'
diner, srate = librosa.load(filename, sr=None)
ipd.Audio(diner, rate=srate)

In [3]:
# Our clip is 32 16th notes long, or 16 8th notes
segs_8 = np.array_split(diner, 16)
# The melody is F#, G#, A, C#, < repeat twice with a syncopated A, then
# F#, C#, B, A, B, A, B, A. I want to grab all unique notes so I can
# rearrange them; I can get all but B in the first 4 8th notes; I just
# counted in my head to get the B.
Fsh, Gsh, A, Csh = segs_8[:4]
B = segs_8[10:11][0]

In [4]:
# I like B, C#, A, A, B, G# as a melody.
tmp_tune = np.hstack((B, Csh, A, A, B, Csh, A, A, A, B, Gsh, Gsh, A, B, Gsh, Gsh))
ipd.display(ipd.Audio(tmp_tune, rate=srate))

In [5]:
# Good, but I want to find an A note with less emphasis.
A2 = segs_8[11:12][0]
composed_tune = np.hstack((B, Csh, A, A2, B, Csh, A, A2, A, B, Gsh, Gsh, A, B, Gsh, Gsh))
ipd.display(ipd.Audio(composed_tune, rate=srate))

In [6]:
# Random swapping of 8th notes
segs_8_shf = np.array_split(diner, 16)
np.random.shuffle(segs_8_shf)
random_tune_8 = np.hstack(segs_8_shf)
sf.write("random_tune_8.wav", random_tune_8 / np.max(np.abs(random_tune_8)), srate)
ipd.display(ipd.Audio(random_tune_8, rate=srate))

In [7]:
# Random swapping of 16th notes
segs_16_shf = np.array_split(diner, 32)
np.random.shuffle(segs_16_shf)
random_tune_16 = np.hstack(segs_16_shf)
sf.write("random_tune_16.wav", random_tune_16 / np.max(np.abs(random_tune_16)), srate)
ipd.display(ipd.Audio(random_tune_16, rate=srate))

In [8]:
# Let's manually get every note.
segs_8 = np.array_split(diner, 16)
segs_16 = np.array_split(diner, 32)
notes = []
for i in range(5):
    notes.append(segs_8[i])
notes.append(segs_16[10])
notes.append(np.hstack((segs_16[11], segs_8[6])))
for i in range(7, 13):
    notes.append(segs_8[i])
notes.append(segs_16[26])
notes.append(np.hstack((segs_16[27], segs_8[14])))
notes.append(segs_8[15])

In [9]:
names = ['F#', 'G#', 'A', 'C#', 'F#', 'G#', 'A', 'C#', 'F#', 'C#', 'B', 'A', 'B', 'A', 'B', 'A']
pitch_of = {'F#': 66, 'G#': 68, 'A': 69, 'B': 71, 'C#': 73}

midi = np.array([pitch_of[n] for n in names])
loud = np.array([np.sqrt(np.mean(n ** 2)) for n in notes])

# bank of notes sorted by loudness
bank = {}
for i in np.argsort(loud):
    bank.setdefault(names[i], []).append(notes[i])

In [10]:
# fade in each segment to avoid clipping
def fade(x, ms=5):
    n = int(srate * ms / 1000)
    ramp = np.linspace(0, 1, n)
    x = x.copy()
    x[:n] *= ramp
    x[len(x) - n:] *= ramp[::-1]
    return x

In [11]:
# remake our original tune
tune = [bank[nm][0] for nm in ['B', 'C#', 'A', 'A', 'B', 'C#', 'A', 'A', 'A', 'B', 'G#', 'G#', 'A', 'B', 'G#', 'G#']]
composed_tune = np.hstack([fade(n) for n in tune])
sf.write("composed_tune.wav", composed_tune / np.max(np.abs(composed_tune)), srate)
ipd.display(ipd.Audio(composed_tune, rate=srate))

In [12]:
# mess around with the fact that we have notes ordered by loudness
rising = np.hstack([fade(notes[i]) for i in np.argsort(midi)])
swell = np.hstack([fade(notes[i]) for i in np.argsort(loud)])
backwards = np.hstack([fade(n) for n in notes[::-1]])

sf.write("rising.wav", rising / np.max(np.abs(rising)), srate)
sf.write("swell.wav", swell / np.max(np.abs(swell)), srate)
sf.write("backwards.wav", backwards / np.max(np.abs(backwards)), srate)

ipd.display(ipd.Audio(rising, rate=srate))
ipd.display(ipd.Audio(swell, rate=srate))
ipd.display(ipd.Audio(backwards, rate=srate))

In [13]:
# swap every note for its closest-sounding other note
feats = np.array([librosa.feature.mfcc(y=n, sr=srate, n_mfcc=13).mean(axis=1) for n in notes])
dist = np.linalg.norm(feats[:, None] - feats[None], axis=2)
np.fill_diagonal(dist, np.inf)
twin = dist.argmin(axis=1)
swapped = np.hstack([fade(notes[i]) for i in twin])

sf.write("swapped.wav", swapped / np.max(np.abs(swapped)), srate)
ipd.display(ipd.Audio(swapped, rate=srate))

In [14]:
# swap each note with another instance of that same note
count = {}
def take(nm):
    count[nm] = count.get(nm, -1) + 1
    return bank[nm][count[nm] % len(bank[nm])]

tune = np.hstack([fade(take(nm)) for nm in names])

sf.write("tune.wav", tune / np.max(np.abs(tune)), srate)
ipd.display(ipd.Audio(tune, rate=srate))

In [15]:
# invert note pitches
order = ['F#', 'G#', 'A', 'B', 'C#']
mirror = dict(zip(order, order[::-1]))
inverted = np.hstack([fade(bank[mirror[nm]][0]) for nm in names])
sf.write("inverted.wav", inverted / np.max(np.abs(inverted)), srate)
ipd.display(ipd.Audio(inverted, rate=srate))

In [16]:
# layering even-indexed segments over odd-indexed segments
def layer(a, b, offset=0):
    out = np.zeros(max(len(a), offset + len(b)))
    out[:len(a)] += fade(a)
    out[offset:offset + len(b)] += fade(b)
    return out

layered = np.hstack([layer(notes[i], notes[i + 1]) for i in range(0, 16, 2)])
layered /= np.abs(layered).max()
sf.write("layered.wav", layered / np.max(np.abs(layered)), srate)
ipd.display(ipd.Audio(layered, rate=srate))